# Papers Topic Hierarchy (Low → Mid → High)

This notebook builds a three-level hierarchy for the existing papers topic model.

Inputs:
- `assets/topic_models/papers_topic_model`
- `assets/topic_models/papers_doc_topics.txt`
- `assets/topic_models/papers_topic_names.txt`
- `assets/embeddings/papers_corpus.txt`
- `assets/synbio_openalex.txt`

Outputs:
- `assets/reports/papers_topic_hierarchy_map.tsv` with columns `ID, low, mid, high`
- `assets/reports/papers_topic_name_hierarchy.tsv` with columns `global_name, low, mid, high`
- `assets/reports/papers_topic_hierarchy_summary.tsv` with mid/high summary stats

In [1]:
from pathlib import Path
import ast

import numpy as np
import pandas as pd
from bertopic import BERTopic
from sklearn.metrics import silhouette_score

SEED = 42
np.random.seed(SEED)

ROOT_DIR = Path.cwd().parent
ASSETS_DIR = ROOT_DIR / "assets"
MODELS_DIR = ASSETS_DIR / "topic_models"
EMBEDDINGS_DIR = ASSETS_DIR / "embeddings"
REPORTS_DIR = ASSETS_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "papers_topic_model"
DOC_TOPICS_PATH = MODELS_DIR / "papers_doc_topics.txt"
TOPIC_NAMES_PATH = MODELS_DIR / "papers_topic_names.txt"
PAPERS_CORPUS_PATH = EMBEDDINGS_DIR / "papers_corpus.txt"
OPENALEX_PATH = ASSETS_DIR / "synbio_openalex.txt"

OUT_DOC_MAP = REPORTS_DIR / "papers_topic_hierarchy_map.tsv"
OUT_NAME_MAP = REPORTS_DIR / "papers_topic_name_hierarchy.tsv"
OUT_SUMMARY = REPORTS_DIR / "papers_topic_hierarchy_summary.tsv"

HIGH_K_MIN = 4
HIGH_K_MAX = 11
MID_K_TARGET = 40

In [2]:
# Load model and input datasets
papers_topic_model = BERTopic.load(str(MODEL_PATH))

papers_doc_topics = pd.read_csv(DOC_TOPICS_PATH, sep="\t")
papers_topic_names = pd.read_csv(TOPIC_NAMES_PATH, sep="\t")
papers_corpus = pd.read_csv(PAPERS_CORPUS_PATH, sep="\t")
papers_openalex = pd.read_csv(OPENALEX_PATH, sep="\t")

# Normalize key column names
papers_doc_topics = papers_doc_topics.rename(columns={"id": "ID", "topic": "low"})
papers_corpus = papers_corpus.rename(columns={"id": "ID"})
papers_openalex = papers_openalex.rename(columns={"id": "ID"})
papers_topic_names = papers_topic_names.rename(columns={"topic": "low"})

required_doc_cols = {"ID", "low"}
required_name_cols = {"low", "global_name"}
required_corpus_cols = {"ID", "text"}
required_openalex_cols = {"ID", "publication_year"}

assert required_doc_cols.issubset(papers_doc_topics.columns), "Missing columns in papers_doc_topics"
assert required_name_cols.issubset(papers_topic_names.columns), "Missing columns in papers_topic_names"
assert required_corpus_cols.issubset(papers_corpus.columns), "Missing columns in papers_corpus"
assert required_openalex_cols.issubset(papers_openalex.columns), "Missing columns in synbio_openalex"

papers_doc_topics["low"] = papers_doc_topics["low"].astype(int)
papers_topic_names["low"] = papers_topic_names["low"].astype(int)
papers_openalex["publication_year"] = pd.to_numeric(papers_openalex["publication_year"], errors="coerce")

print(f"papers_doc_topics: {len(papers_doc_topics):,} rows")
print(f"papers_topic_names: {len(papers_topic_names):,} rows")
print(f"papers_corpus: {len(papers_corpus):,} rows")
print(f"papers_openalex: {len(papers_openalex):,} rows")
print(f"Outlier docs (low = -1): {(papers_doc_topics['low'] == -1).sum():,}")

papers_doc_topics: 24,202 rows
papers_topic_names: 263 rows
papers_corpus: 24,202 rows
papers_openalex: 24,202 rows
Outlier docs (low = -1): 9,017


In [3]:
# Build BERTopic-native hierarchy once, then cut it at different levels

docs_for_hierarchy = papers_corpus["text"].astype(str).tolist()
if len(docs_for_hierarchy) != len(papers_topic_model.topics_):
    raise ValueError(
        "papers_corpus length does not match model topics length. "
        "hierarchical_topics requires training-aligned docs."
    )

hier_df = papers_topic_model.hierarchical_topics(docs_for_hierarchy, use_ctfidf=False)

# Auto-detect expected hierarchy columns across BERTopic versions
col_parent = "Parent_ID" if "Parent_ID" in hier_df.columns else "Parent Topic"
col_left = "Child_Left_ID" if "Child_Left_ID" in hier_df.columns else "Child Left Topic"
col_right = "Child_Right_ID" if "Child_Right_ID" in hier_df.columns else "Child Right Topic"
col_distance = "Distance"

if col_distance not in hier_df.columns:
    raise ValueError("Could not find 'Distance' column in hierarchical topics dataframe")

# Non-outlier low topics that appear in the document assignment table
low_topics = sorted([t for t in papers_doc_topics["low"].unique() if t >= 0])

# Helper: convert merge-tree rows to low-topic -> cluster map for a target k
merges = hier_df[[col_parent, col_left, col_right, col_distance]].copy().sort_values(col_distance)

node_members = {int(t): {int(t)} for t in low_topics}
active_nodes = set(low_topics)
maps_by_k = {}

max_k = max(max(HIGH_K_MAX, MID_K_TARGET), len(low_topics))
tracked_ks = {k for k in range(1, len(low_topics) + 1) if k <= max_k}

for _, row in merges.iterrows():
    left = int(row[col_left])
    right = int(row[col_right])
    parent = int(row[col_parent])

    if left not in node_members or right not in node_members:
        continue
    if left not in active_nodes or right not in active_nodes:
        continue

    parent_members = node_members[left] | node_members[right]
    node_members[parent] = parent_members

    active_nodes.remove(left)
    active_nodes.remove(right)
    active_nodes.add(parent)

    k_now = len(active_nodes)
    if k_now in tracked_ks and k_now not in maps_by_k:
        clusters = [sorted(node_members[n]) for n in active_nodes]
        clusters = sorted(clusters, key=lambda c: (min(c), len(c)))
        topic_to_group = {}
        for group_id, members in enumerate(clusters):
            for topic in members:
                topic_to_group[int(topic)] = int(group_id)
        maps_by_k[k_now] = topic_to_group

if MID_K_TARGET not in maps_by_k:
    MID_K_TARGET = max(k for k in maps_by_k.keys() if k >= HIGH_K_MAX)

# Topic embeddings for quality scoring of high-level cuts
model_topics_sorted = sorted([int(t) for t in papers_topic_model.get_topics().keys()])
if papers_topic_model.topic_embeddings_ is None:
    raise ValueError("Model does not contain topic embeddings required for hierarchy scoring")
if len(model_topics_sorted) != papers_topic_model.topic_embeddings_.shape[0]:
    raise ValueError("Mismatch between topic IDs and topic embeddings rows")

topic_to_embedding = {
    topic_id: papers_topic_model.topic_embeddings_[idx]
    for idx, topic_id in enumerate(model_topics_sorted)
}

embed_topics = [t for t in low_topics if t in topic_to_embedding]
X_low = np.vstack([topic_to_embedding[t] for t in embed_topics])

candidate_scores = []
for k in range(HIGH_K_MIN, HIGH_K_MAX + 1):
    if k not in maps_by_k:
        continue
    labels = [maps_by_k[k][t] for t in embed_topics]
    if len(set(labels)) < 2:
        continue
    score = silhouette_score(X_low, labels, metric="cosine")
    candidate_scores.append((k, score))

if not candidate_scores:
    raise ValueError("Could not score high-level hierarchy candidates in requested range")

selected_high_k, selected_high_score = max(candidate_scores, key=lambda x: x[1])

low_to_mid = maps_by_k[MID_K_TARGET]
low_to_high = maps_by_k[selected_high_k]

topic_hierarchy_map = pd.DataFrame({"low": low_topics})
topic_hierarchy_map["mid"] = topic_hierarchy_map["low"].map(low_to_mid).astype(int)
topic_hierarchy_map["high"] = topic_hierarchy_map["low"].map(low_to_high).astype(int)

print(f"Low topics: {len(low_topics)}")
print(f"Mid groups (target): {MID_K_TARGET}")
print(f"High groups (selected): {selected_high_k} | silhouette={selected_high_score:.4f}")
pd.DataFrame(candidate_scores, columns=["high_k", "silhouette"]).sort_values("high_k")

100%|██████████| 262/262 [00:10<00:00, 26.17it/s]


Low topics: 263
Mid groups (target): 40
High groups (selected): 11 | silhouette=0.1733


,high_k,silhouette
0,4,0.148989
1,5,0.146081
2,6,0.139180
3,7,0.148013
4,8,0.142613
5,9,0.156006
6,10,0.164337
7,11,0.173264


In [4]:
# Report 1: paper-level mapping (ID, low, mid, high)

doc_map = papers_doc_topics.merge(topic_hierarchy_map, on="low", how="left")

doc_map["mid"] = doc_map["mid"].fillna(-1).astype(int)
doc_map["high"] = doc_map["high"].fillna(-1).astype(int)

report1 = doc_map[["ID", "low", "mid", "high"]].copy()
report1.to_csv(OUT_DOC_MAP, sep="\t", index=False)

print(f"Saved: {OUT_DOC_MAP}")
report1.head()

Saved: /Users/cristian/Desktop/GitHub/igem-synbio/assets/reports/papers_topic_hierarchy_map.tsv


,ID,low,mid,high
0,https://openalex.org/W2072812562,167,24,5
1,https://openalex.org/W2091309425,167,24,5
2,https://openalex.org/W2037457710,44,16,5
3,https://openalex.org/W1984248485,7,6,0
4,https://openalex.org/W1980762198,65,27,3


In [5]:
# Report 2: low-level global names mapped to low/mid/high

report2 = papers_topic_names.merge(topic_hierarchy_map, on="low", how="left")
report2["mid"] = report2["mid"].fillna(-1).astype(int)
report2["high"] = report2["high"].fillna(-1).astype(int)
report2 = report2[["global_name", "low", "mid", "high"]].sort_values("low")
report2.to_csv(OUT_NAME_MAP, sep="\t", index=False)

print(f"Saved: {OUT_NAME_MAP}")
report2.head()

Saved: /Users/cristian/Desktop/GitHub/igem-synbio/assets/reports/papers_topic_name_hierarchy.tsv


,global_name,low,mid,high
0,Genetic Circuit Control,0,0,0
1,Synthetic Nucleic Acid Design,1,1,1
2,Plant Biosynthetic Pathways,2,2,2
3,Synthetic CpG ODNs in Immunity,3,3,3
4,Ethics and Governance in Synthetic Biology,4,4,4


In [6]:
# Report 3: mid/high summaries with count, average year, median year

doc_map_year = report1.merge(
    papers_openalex[["ID", "publication_year"]],
    on="ID",
    how="left",
)

mid_summary = (
    doc_map_year[doc_map_year["mid"] >= 0]
    .groupby("mid", as_index=False)
    .agg(
        total_count=("ID", "count"),
        avg_publication_year=("publication_year", "mean"),
        median_publication_year=("publication_year", "median"),
    )
    .rename(columns={"mid": "group_id"})
)
mid_summary.insert(0, "level", "mid")

high_summary = (
    doc_map_year[doc_map_year["high"] >= 0]
    .groupby("high", as_index=False)
    .agg(
        total_count=("ID", "count"),
        avg_publication_year=("publication_year", "mean"),
        median_publication_year=("publication_year", "median"),
    )
    .rename(columns={"high": "group_id"})
)
high_summary.insert(0, "level", "high")

report3 = pd.concat([mid_summary, high_summary], ignore_index=True)
report3 = report3.sort_values(["level", "group_id"]).reset_index(drop=True)
report3.to_csv(OUT_SUMMARY, sep="\t", index=False)

print(f"Saved: {OUT_SUMMARY}")
report3.head(10)

Saved: /Users/cristian/Desktop/GitHub/igem-synbio/assets/reports/papers_topic_hierarchy_summary.tsv


,level,group_id,total_count,avg_publication_year,median_publication_year
0,high,0,2603,2016.815597,2018.0
1,high,1,1802,2003.721421,2005.0
2,high,2,2024,2017.669960,2020.0
3,high,3,1154,1998.142114,1997.0
4,high,4,1281,2016.663544,2017.0
5,high,5,2317,2010.947346,2016.0
6,high,6,1686,2015.708778,2019.0
7,high,7,508,2015.834646,2020.0
8,high,8,1128,2005.164007,2006.0
9,high,9,554,2007.516245,2011.0


In [7]:
# Final validation checks

assert list(report1.columns) == ["ID", "low", "mid", "high"], "Report 1 columns mismatch"
assert list(report2.columns) == ["global_name", "low", "mid", "high"], "Report 2 columns mismatch"
assert set(["level", "group_id", "total_count", "avg_publication_year", "median_publication_year"]).issubset(report3.columns), "Report 3 columns mismatch"

assert len(report1) == len(papers_doc_topics), "Report 1 row count mismatch"
assert (report1.loc[report1["low"] == -1, ["mid", "high"]] == -1).all().all(), "Outlier mapping mismatch"
assert HIGH_K_MIN <= selected_high_k <= HIGH_K_MAX, "Selected high-level K out of requested range"

print("All validation checks passed.")
print(f"Rows -> report1: {len(report1):,}, report2: {len(report2):,}, report3: {len(report3):,}")

All validation checks passed.
Rows -> report1: 24,202, report2: 263, report3: 51


In [8]:
# Compact hierarchy quality summary

quality_summary = pd.DataFrame(candidate_scores, columns=["high_k", "silhouette"]).sort_values("high_k")
outlier_pct = (papers_doc_topics["low"] == -1).mean() * 100

print(f"Selected high-level K: {selected_high_k}")
print(f"Selected silhouette: {selected_high_score:.4f}")
print(f"Outlier documents (low = -1): {outlier_pct:.2f}%")

quality_summary

Selected high-level K: 11
Selected silhouette: 0.1733
Outlier documents (low = -1): 37.26%


,high_k,silhouette
0,4,0.148989
1,5,0.146081
2,6,0.139180
3,7,0.148013
4,8,0.142613
5,9,0.156006
6,10,0.164337
7,11,0.173264
